# ML-06 — Signal Audit: Do the Flags Hold?

## 1. Distributions

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
ROOT = Path.cwd()
while ROOT.name and not (ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv'
df = pd.read_csv(DATA)
print('Rows:', len(df), '| Columns:', len(df.columns))
for c in ['impressions_90d','days_since_last_update','ctr','avg_position','word_count']:
    print(c, 'median=', df[c].median(), 'p95=', df[c].quantile(.95))

## 2. Signal tests
We compare the observed decline rate across useful bins.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
ROOT = Path.cwd()
while ROOT.name and not (ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv'
df = pd.read_csv(DATA)
print('Rows:', len(df), '| Columns:', len(df.columns))
y=df.trend_direction.eq('down')
for c in ['impressions_90d','avg_position','ctr','days_since_last_update']:
    bins=pd.qcut(df[c],4,duplicates='drop')
    rates=df.assign(declining=y).groupby(bins,observed=True).declining.mean()
    print('\n',c); print(rates.round(3))

## 3. Verdicts
- **Volume: CONFIRMED** — decline rates are substantially higher in moderate/high-impression bins than the lowest bin.
- **CTR vs position: CONFIRMED** — among visible pages, declining pages have lower median CTR than non-declining pages.
- **Staleness: OPPOSITE** — the simple `days_since_last_update >= 180` assumption is not supported as a stronger decline flag in this snapshot; its decline rate is lower than the non-stale group.

These are directional associations, not causal findings.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
ROOT = Path.cwd()
while ROOT.name and not (ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv'
df = pd.read_csv(DATA)
print('Rows:', len(df), '| Columns:', len(df.columns))
visible=df[(df.impressions_90d>=500)&(df.avg_position<=20)].copy()
print('Visible-page rows:',len(visible))
print(visible.groupby(visible.trend_direction.eq('down')).ctr.median().rename({False:'not_declining',True:'declining'}))
print(pd.crosstab(df.days_since_last_update>=180,y,normalize='index').round(3))

## 4. Practical meaning
Prioritize measurable visibility and CTR opportunity signals, but do not treat staleness alone as evidence of decline. The baseline remains useful as a transparent comparator rather than as a source of truth.

## Self-check
- [x] Three safe signals tested
- [x] Flag-linked staleness assumption challenged
- [x] No causal language